In [6]:
import os
import cv2
import csv
import math
import time
import random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import kagglehub
from tqdm import tqdm


# ============================================================
# 1. Download / locate dataset
# ============================================================
print("\n[1/6] Downloading or locating FaceForensics++ dataset...")
ffpp_path = kagglehub.dataset_download("xdxd003/ff-c23")
print(f"[1/6] Dataset ready at: {ffpp_path}")

FFPP_RAW_ROOT = Path(ffpp_path)

# ============================================================
# 2. Output config
# ============================================================
OUTPUT_ROOT = FFPP_RAW_ROOT / "ffpp_preprocessed"
FRAMES_DIR = OUTPUT_ROOT / "all_frames"
METADATA_CSV = OUTPUT_ROOT / "metadata.csv"

FRAMES_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

# Speed / quality tradeoff
TARGET_FPS = 2
MAX_FRAMES_PER_VIDEO = 8
JPEG_QUALITY = 95

# Split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Reproducibility
SEED = 42

# Workers
NUM_WORKERS = max(1, min(os.cpu_count() or 4, 8))


# ============================================================
# 3. Helpers
# ============================================================
def list_all_videos(root: Path):
    print("\n[2/6] Scanning dataset folders for videos...")
    videos = []
    for p in tqdm(root.rglob("*"), desc="Scanning files", ncols=100):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            videos.append(p)
    videos = sorted(videos)
    print(f"[2/6] Found {len(videos)} video files.")
    return videos


def infer_label_from_path(path: Path):
    """
    Return:
      0 for real
      1 for fake
    """
    s = str(path).lower()

    real_keywords = [
        "original",
        "original_sequences",
        "youtube",
        "pristine",
        "real",
    ]

    fake_keywords = [
        "manipulated",
        "manipulated_sequences",
        "deepfakes",
        "faceswap",
        "face2face",
        "neuraltextures",
        "fake",
    ]

    if any(k in s for k in real_keywords):
        return 0
    if any(k in s for k in fake_keywords):
        return 1

    return 1


def build_video_list(root: Path):
    print("\n[3/6] Building labeled video list...")
    video_paths = list_all_videos(root)

    items = []
    real_count = 0
    fake_count = 0

    for v in tqdm(video_paths, desc="Labelling videos", ncols=100):
        label = infer_label_from_path(v)
        items.append((v, label))
        if label == 0:
            real_count += 1
        else:
            fake_count += 1

    print(f"[3/6] Label summary -> Real: {real_count}, Fake: {fake_count}")
    return items


def stratified_split(items, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8

    print("\n[4/6] Creating stratified train/val/test split...")

    rng = random.Random(seed)

    real_items = [x for x in items if x[1] == 0]
    fake_items = [x for x in items if x[1] == 1]

    rng.shuffle(real_items)
    rng.shuffle(fake_items)

    def split_class(class_items):
        n = len(class_items)
        n_train = int(train_ratio * n)
        n_val = int(val_ratio * n)
        n_test = n - n_train - n_val

        train = class_items[:n_train]
        val = class_items[n_train:n_train + n_val]
        test = class_items[n_train + n_val:]
        return train, val, test

    real_train, real_val, real_test = split_class(real_items)
    fake_train, fake_val, fake_test = split_class(fake_items)

    train = real_train + fake_train
    val = real_val + fake_val
    test = real_test + fake_test

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)

    print(f"[4/6] Train videos: {len(train)}")
    print(f"[4/6] Val videos:   {len(val)}")
    print(f"[4/6] Test videos:  {len(test)}")

    return train, val, test


def safe_rel_video_id(video_path: Path, root: Path):
    rel = video_path.relative_to(root)
    rel_no_suffix = rel.with_suffix("")
    return "__".join(rel_no_suffix.parts)


def extract_frames_from_video(
    video_path_str,
    root_str,
    frames_dir_str,
    split_name,
    label,
    target_fps=2,
    max_frames=8,
    jpeg_quality=95,
):
    """
    Worker function for multiprocessing.
    Saves frames into one flat folder and returns metadata rows.
    """
    video_path = Path(video_path_str)
    root = Path(root_str)
    frames_dir = Path(frames_dir_str)

    video_id = safe_rel_video_id(video_path, root)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "video_path": str(video_path),
            "video_id": video_id,
            "label": label,
            "split": split_name,
            "saved_frames": 0,
            "rows": [],
            "error": "Could not open video",
        }

    vid_fps = cap.get(cv2.CAP_PROP_FPS)
    if vid_fps <= 0 or math.isnan(vid_fps):
        vid_fps = 25.0

    frame_interval = max(int(round(vid_fps / target_fps)), 1)

    frame_idx = 0
    saved = 0
    rows = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            frame_name = f"{split_name}_{label}_{video_id}_{saved:04d}.jpg"
            out_path = frames_dir / frame_name

            ok = cv2.imwrite(
                str(out_path),
                frame,
                [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality]
            )

            if ok:
                rows.append({
                    "frame_path": str(out_path),
                    "frame_name": frame_name,
                    "video_path": str(video_path),
                    "video_id": video_id,
                    "split": split_name,
                    "label": label,
                    "label_name": "Real" if label == 0 else "Fake",
                    "frame_index_in_video": frame_idx,
                    "sample_index": saved,
                })
                saved += 1

            if max_frames is not None and saved >= max_frames:
                break

        frame_idx += 1

    cap.release()

    return {
        "video_path": str(video_path),
        "video_id": video_id,
        "label": label,
        "split": split_name,
        "saved_frames": saved,
        "rows": rows,
        "error": "",
    }


def save_metadata(rows, csv_path: Path):
    print("\n[6/6] Writing metadata CSV...")

    csv_path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "frame_path",
        "frame_name",
        "video_path",
        "video_id",
        "split",
        "label",
        "label_name",
        "frame_index_in_video",
        "sample_index",
    ]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"[6/6] Metadata saved to: {csv_path}")


# ============================================================
# 4. Main
# ============================================================
if __name__ == "__main__":
    start_time = time.time()
    print("\nStarting FF++ preprocessing...\n")

    all_videos = build_video_list(FFPP_RAW_ROOT)

    train_videos, val_videos, test_videos = stratified_split(
        all_videos,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        seed=SEED,
    )

    jobs = []
    for split_name, split_items in [
        ("Train", train_videos),
        ("Val", val_videos),
        ("Test", test_videos),
    ]:
        for video_path, label in split_items:
            jobs.append((
                str(video_path),
                str(FFPP_RAW_ROOT),
                str(FRAMES_DIR),
                split_name,
                label,
                TARGET_FPS,
                MAX_FRAMES_PER_VIDEO,
                JPEG_QUALITY,
            ))

    print("\n[5/6] Beginning frame extraction...")
    print(f"[5/6] Output folder: {FRAMES_DIR}")
    print(f"[5/6] Workers: {NUM_WORKERS}")
    print(f"[5/6] Target FPS: {TARGET_FPS}")
    print(f"[5/6] Max frames/video: {MAX_FRAMES_PER_VIDEO}")
    print(f"[5/6] Total videos to process: {len(jobs)}")

    all_rows = []
    failed = []
    total_frames_saved = 0

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = [executor.submit(extract_frames_from_video, *job) for job in jobs]

        with tqdm(total=len(futures), desc="Extracting videos", ncols=100) as pbar:
            for i, fut in enumerate(as_completed(futures), start=1):
                result = fut.result()

                if result["error"]:
                    failed.append(result)

                all_rows.extend(result["rows"])
                total_frames_saved += result["saved_frames"]

                pbar.update(1)
                pbar.set_postfix({
                    "frames": total_frames_saved,
                    "failed": len(failed),
                })

                # occasional friendly status line
                if i % 100 == 0 or i == len(futures):
                    print(
                        f"[5/6] Processed {i}/{len(futures)} videos | "
                        f"Frames saved: {total_frames_saved} | Failed: {len(failed)}"
                    )

    save_metadata(all_rows, METADATA_CSV)

    elapsed = time.time() - start_time

    print("\nPreprocessing complete.")
    print(f"Frames folder:   {FRAMES_DIR}")
    print(f"Metadata CSV:    {METADATA_CSV}")
    print(f"Total frames:    {len(all_rows)}")
    print(f"Failed videos:   {len(failed)}")
    print(f"Elapsed time:    {elapsed / 60:.2f} minutes")


[1/6] Downloading or locating FaceForensics++ dataset...
[1/6] Dataset ready at: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1

Starting FF++ preprocessing...


[3/6] Building labeled video list...

[2/6] Scanning dataset folders for videos...


Scanning files: 52504it [00:29, 1752.23it/s]  


[2/6] Found 7000 video files.


Labelling videos: 100%|█████████████████████████████████████| 7000/7000 [00:00<00:00, 637363.03it/s]

[3/6] Label summary -> Real: 1000, Fake: 6000

[4/6] Creating stratified train/val/test split...
[4/6] Train videos: 5600
[4/6] Val videos:   700
[4/6] Test videos:  700

[5/6] Beginning frame extraction...
[5/6] Output folder: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames
[5/6] Workers: 8
[5/6] Target FPS: 2
[5/6] Max frames/video: 8
[5/6] Total videos to process: 7000


...

[5/6] Processed 6800/7000 videos | Frames saved: 54353 | Failed: 0


Extracting videos:  99%|███████████████▊| 6901/7000 [34:22<00:37,  2.67it/s, frames=55161, failed=0]

[5/6] Processed 6900/7000 videos | Frames saved: 55153 | Failed: 0


Extracting videos: 100%|████████████████| 7000/7000 [34:52<00:00,  3.35it/s, frames=55953, failed=0]


[5/6] Processed 7000/7000 videos | Frames saved: 55953 | Failed: 0

[6/6] Writing metadata CSV...
[6/6] Metadata saved to: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv

Preprocessing complete.
Frames folder:   /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames
Metadata CSV:    /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv
Total frames:    55953
Failed videos:   0
Elapsed time:    35.38 minutes


In [4]:
!pip install opencv-python-headless kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 78.4 kB/s eta 0:00:00a 0:00:01
  Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 96.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 629.3 kB/s eta 0:00:00a 0:00:01
Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl (324 kB)
